# Cross-dataset check: VISOB-trained models on UBIPr (zero-shot)

**What this measures.** Our models were trained on VISOB: eye crops taken with smartphones (front cameras, at arm's length). This notebook asks how well such a model recognises eyes from a completely different source, without any retraining (zero-shot): **UBIPr** (UBI Periocular), photographed with a Canon EOS 5D DSLR in visible light at 4 to 8 metres. It is cross-dataset *and* cross-sensor (smartphone to DSLR), with people the model has never seen. A good result means the model learned eye features that carry over; a poor result shows how much it depends on VISOB's capture conditions.

**How.** `scripts/20_prepare_ubipr.py` cuts every UBIPr image to a VISOB-like eye crop (240 x 160, framed from the annotated eye corners so the eye fills the width as in VISOB) and writes a manifest. `scripts/15_eval_verification.py --modes x1` then compares embeddings at the ArcFace input: each class (one eye of one person) is enrolled as the mean of its session-1 images, and its session-2 images are the probes. Genuine = same class; impostor = another person, same eye side. Reported: EER, TAR at FAR 1 % and 0.1 %, rank-1, with 95 % CIs from a bootstrap over persons, overall and per camera distance and glasses.

**Licence (read before running).** UBIPr is licensed CC BY-NC-SA 4.0: non-commercial use only. Any paper or report that uses it must cite C. Padole and H. Proença, "Periocular recognition: Analysis of performance degradation factors", ICB 2012, and UBIRIS.v2 (H. Proença, S. Filipe, R. Santos, J. Oliveira, L. Alexandre, "The UBIRIS.v2: A database of visible wavelength iris images captured on-the-move and at-a-distance", IEEE TPAMI 2010). **Never publish the images or the crops**: keep this notebook and its outputs private, and `results.zip` holds no images (only numbers, the manifest and the ROC plot).

**Settings.** Accelerator: GPU optional (a T4 is about 10x faster than the CPU). Internet **on** (the archive is downloaded from https://iris.di.ubi.pt/ubipr.tar). Secret `github_token` attached (the repository is private). Input: `nandininautiyal/ocular-xception-arcface-full-baseline`. No VISOB input is needed.

**Expected time** (estimates: the crops took 2.3 min on the laptop; nothing has been timed on Kaggle yet, and the smoke run in cell 7 prints this session's estimate). Download 1-3 min (491 MB), crops 3-5 min (10,199 images), then per checkpoint roughly 5 min on a T4 or 30-40 min on the CPU: model loading, embeddings of about 10,200 crops, bootstrap. With one checkpoint: about 15 min on a GPU.

**Cells.** 1 setup, 2 settings (the only cell to edit; add later models to `CKPTS`), 3 commands and pre-flight checks, 4 download, 5 crops, 6 QC sheet, 7 verification per checkpoint, 8 results, 9 `results.zip`, 10 final checks.

In [ ]:
# 1. Setup: clone the repo at BRANCH, install the pinned tf-keras, record the commit
import os, subprocess, time
SESSION_START = time.time()
BRANCH = 'main'   # must contain scripts/20_prepare_ubipr.py and 15_eval_verification.py with --modes x1

!pip install -q --no-deps tf-keras==2.20.1

from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret('github_token')
REPO = '/tmp/ocular-adv-xai'
url = f'https://{tok}@github.com/nandininautiyal/ocular-adv-xai.git'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
    !git -C {REPO} remote set-url origin https://github.com/nandininautiyal/ocular-adv-xai.git
else:
    # a reused session may have another branch checked out: always land exactly on BRANCH
    !git -C {REPO} fetch -q {url} {BRANCH}
    !git -C {REPO} checkout -q -f -B {BRANCH} FETCH_HEAD
%cd {REPO}
!git log -3 --oneline
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('branch:', BRANCH, '| commit:', COMMIT)
!nvidia-smi -L || echo 'no GPU: the embeddings run on the CPU (slower, same results)'

In [ ]:
# 2. Settings: the only cell to edit
# checkpoints to evaluate, label -> path; add later models here (each gets its own outputs, tag ubipr_<label>)
CKPTS = {
    'full': '/kaggle/input/datasets/nandininautiyal/ocular-xception-arcface-full-baseline/trained_models/'
            'Xception_ArcFace_full.keras',
}
UBIPR_URL = 'https://iris.di.ubi.pt/ubipr.tar'
# sha256 and size of the copy checked on the laptop on 1 Nov 2026 (10,199 images); a different file is reported,
# and its counts are then checked below
KNOWN_SHA256 = '5bc959d359d2cea4ab5c20fdf970d4197732bd9c36aa40afa8118bc62bf7b4cc'
KNOWN_SIZE = 491376128
N_BOOT = 1000        # bootstrap resamples for the CIs
BATCH = 64           # images per batch in 15; lower to 32 if the GPU runs out of memory
WORKERS = 4          # threads reading crops in 15
SMOKE = True         # first a 10-person smoke run of 15 with the first checkpoint (about 2 min)

W = '/kaggle/working'
UBIPR_TAR = f'{W}/ubipr.tar'
UBIPR = f'{W}/ubipr'                      # 20's crops (never download or publish them), manifest and summary
OUT = f'{W}/verification_ubipr'           # 15's outputs
LOGS = f'{W}/logs'
os.makedirs(LOGS, exist_ok=True)

## 3. Commands and pre-flight checks

Every command of the notebook is built here. The check runs each script's `--help` in its own process and stops if a flag used below is missing (e.g. `BRANCH` lacks the x1 mode), and it finds each checkpoint (searching `/kaggle/input` by file name if the path has moved).

In [ ]:
# 3. Commands, then pre-flight: scripts and flags (argparse, separate processes), checkpoints, disk
import glob, hashlib, json, shutil
import pandas as pd

def sh(cmd, check=True, log=None):
    # run cmd from the repo with its output shown here (and saved to logs/<log>.log); check: stop on failure
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, executable='/bin/bash', text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = p.wait()
    if log:
        with open(f'{LOGS}/{log}.log', 'w') as f:
            f.writelines(lines)
    assert code == 0 or not check, f'exit code {code}: {cmd}'
    return code

def load(path):
    with open(path) as f:
        return json.load(f)

for label, path in list(CKPTS.items()):
    if not os.path.exists(path):
        found = sorted(glob.glob(f'/kaggle/input/**/{os.path.basename(path)}', recursive=True))
        print(f'{label}: {path} not found;', f'using {found[0]}' if found else 'no copy under /kaggle/input')
        if found:
            CKPTS[label] = found[0]

verify = (f'--modes x1 --external-manifest {UBIPR}/ubipr_manifest.csv --external-name ubipr '
          f'--external-subsets cam_distance,glasses --batch-size {BATCH} --workers {WORKERS} --out-dir {OUT}')
CMD = {'20_prepare': f'python -u scripts/20_prepare_ubipr.py --source {UBIPR_TAR} --out-dir {UBIPR}'}
first = next(iter(CKPTS))
if SMOKE:
    CMD['15_smoke'] = (f"python -u scripts/15_eval_verification.py --ckpt '{CKPTS[first]}' {verify} "
                       f'--max-subjects 10 --n-boot 100 --tag ubipr_smoke')
for label, path in CKPTS.items():
    CMD[f'15_{label}'] = (f"python -u scripts/15_eval_verification.py --ckpt '{path}' {verify} --n-boot {N_BOOT} "
                          f'--tag ubipr_{label}')

problems = []
for script in sorted({t for c in CMD.values() for t in c.split() if t.startswith('scripts/')}):
    r = subprocess.run(['python', script, '--help'], cwd=REPO, capture_output=True, text=True,
                       env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
    if r.returncode != 0:
        problems.append(f'{script} --help failed: {(r.stderr or r.stdout).strip()[-300:]}')
        continue
    used = {t.split('=')[0] for c in CMD.values() if script in c.split()
            for t in c.split()[c.split().index(script) + 1:] if t.startswith('--')}
    lost = sorted(f for f in used if f not in r.stdout)
    if lost:
        problems.append(f'{script} has no {lost}: BRANCH lacks the cross-dataset code')
    else:
        print(f'{script}: all {len(used)} flags exist')
for label, path in CKPTS.items():
    if not os.path.exists(path):
        problems.append(f'checkpoint {label} not found: {path} (add the input dataset)')
free_gb = shutil.disk_usage(W).free / 2**30
print(f'free disk: {free_gb:.1f} GB (the archive and the crops need about 1.5 GB)')
if free_gb < 2:
    problems.append(f'only {free_gb:.1f} GB free in {W}')
for name, c in CMD.items():
    print(f'{name}: {c}')
assert not problems, 'fix these first:\n- ' + '\n- '.join(problems)

## 4. Download

The archive comes straight from the dataset's site (Internet on). Its sha256 and size are recorded in `ubipr_download.json` (part of `results.zip`), and compared with the copy checked on the laptop.

In [ ]:
# 4. Download ubipr.tar (skipped if already here) and record its sha256 and size
t0 = time.time()
if not os.path.exists(UBIPR_TAR):
    sh(f'curl -L --fail --retry 3 --retry-delay 10 -sS -o {UBIPR_TAR}.part {UBIPR_URL} && mv {UBIPR_TAR}.part {UBIPR_TAR}')
h = hashlib.sha256()
with open(UBIPR_TAR, 'rb') as f:
    for block in iter(lambda: f.read(1 << 22), b''):
        h.update(block)
DOWNLOAD = {'url': UBIPR_URL, 'sha256': h.hexdigest(), 'size_bytes': os.path.getsize(UBIPR_TAR),
            'same_as_laptop_copy': h.hexdigest() == KNOWN_SHA256 and os.path.getsize(UBIPR_TAR) == KNOWN_SIZE,
            'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()), 'seconds': round(time.time() - t0, 1)}
with open(f'{W}/ubipr_download.json', 'w') as f:
    json.dump(DOWNLOAD, f, indent=2)
print(json.dumps(DOWNLOAD, indent=2))
if not DOWNLOAD['same_as_laptop_copy']:
    print('NOTE: not the file checked on 1 Nov 2026 (sha256 or size differs); the counts are checked in cell 5')

## 5-6. Crops and the QC sheet

20 reads the tar directly (no extraction), parses every annotation, cuts the crops and writes `ubipr/ubipr_manifest.csv`, `ubipr_skipped.csv` and `ubipr_summary.json`. On the 1 Nov copy: 10,199 images, 1 skipped (an eye corner outside the image), 10,198 crops, of which 37 are marked unusable (3 whose eye label disagrees with their class, 24 session-1 images taken on their class's session-2 date, 12 misfiled crops whose photograph lies outside their person's capture run, 2 of them also with the wrong eye label), 507 classes, 254 persons, 2,642 probes. Look at the QC sheet: the crops should frame the eye like VISOB (the eye corners near the left and right edges, the eye a little above the middle).

In [ ]:
# 5. Crops and manifest (about 3 min)
t0 = time.time()
sh(CMD['20_prepare'], log='20_prepare')
PREP = load(f'{UBIPR}/ubipr_summary.json')
print(f'20: {(time.time() - t0) / 60:.1f} min')
for key in ('files', 'n_crops', 'n_usable', 'n_skipped', 'skipped_by_reason', 'flags', 'n_classes', 'n_persons',
            'misfiled', 'cross_person_links', 'images_by_session', 'images_by_cam_distance', 'images_by_glasses', 'x1_protocol',
            'crop'):
    print(f'{key}: {PREP[key]}')

In [ ]:
# 6. QC contact sheet: 48 random crops (class, eye, session, distance); private, never publish it
from IPython.display import Image, display
display(Image(filename=f'{UBIPR}/qc_contact_sheet.png', width=1100))

## 7-8. Verification per checkpoint and the results

First a 10-person smoke run (the first checkpoint, 100 bootstrap resamples), then the full x1 run of every checkpoint in `CKPTS`. Each writes `verification_ubipr/verification_metrics_ubipr_<label>.csv` (rows: all, cam_distance=4..8, glasses=No/Yes), the score sample, the ROC / DET plot and the summary JSON. `n_subjects` counts persons; FAR and FRR are per comparison.

In [ ]:
# 7. 15 --modes x1: smoke run, then every checkpoint
STATUS, t0 = {}, time.time()
if SMOKE:
    STATUS['15_smoke'] = sh(CMD['15_smoke'], check=False, log='15_smoke')
    assert STATUS['15_smoke'] == 0, 'the smoke run failed: see its output above'
    s = load(f'{OUT}/verification_summary_ubipr_smoke.json')
    t = s['timing']
    per_image = t['embedding_x1_s'] / s['external']['n_images_embedded']
    print(f"smoke: {s['external']['n_images_embedded']} crops in {t['embedding_x1_s']} s; full run estimate per "
          f"checkpoint: {(per_image * PREP['n_usable'] + t['load_model_s'] + 60) / 60:.0f} min")
for label in CKPTS:
    STATUS[f'15_{label}'] = sh(CMD[f'15_{label}'], check=False, log=f'15_{label}')
print(f'verification: {(time.time() - t0) / 60:.1f} min | exit codes: {STATUS}')

In [ ]:
# 8. Results: overall and per subset, for every checkpoint (CIs: 95 %, bootstrap over persons)
pd.set_option('display.width', 200)
cols = ['subset', 'n_subjects', 'n_probes', 'n_genuine', 'n_impostor', 'eer', 'eer_ci_low', 'eer_ci_high',
        'tar_at_far_1e-2', 'tar_at_far_1e-3', 'tar_at_far_1e-3_ci_low', 'tar_at_far_1e-3_ci_high', 'rank1',
        'rank1_ci_low', 'rank1_ci_high']
TABLE = []
for label in CKPTS:
    path = f'{OUT}/verification_metrics_ubipr_{label}.csv'
    if not os.path.exists(path):
        print(f'{label}: no results (exit code {STATUS.get(f"15_{label}")})')
        continue
    m = pd.read_csv(path)
    TABLE.append(m.assign(model=label))
    print(f'\n=== {label}: zero-shot UBIPr (x1) ===')
    print(m[cols].round(4).to_string(index=False))
    print('caveat:', m.caveat.iloc[0])
    png = f'{OUT}/verification_roc_det_ubipr_{label}.png'
    if os.path.exists(png):
        display(Image(filename=png, width=900))
if TABLE:
    overall = pd.concat(TABLE)
    print('\nall checkpoints, overall:')
    print(overall[overall.subset == 'all'][['model'] + cols[1:]].round(4).to_string(index=False))

## 9-10. results.zip and the final checks

`results.zip` holds numbers only: the download record, 20's manifest (landmarks and attributes, no pixels), skipped list and summary, 15's metrics, score samples, summaries and ROC / DET plots, and the logs. The crops and the QC sheet stay out. The final cell checks the counts and every output.

In [ ]:
# 9. results.zip (download it from the Output panel; never commit or upload it)
import zipfile
info = {'branch': BRANCH, 'commit': COMMIT, 'checkpoints': CKPTS, 'commands': CMD, 'status': STATUS,
        'n_boot': N_BOOT, 'download': DOWNLOAD, 'session_minutes': round((time.time() - SESSION_START) / 60, 1)}
with open(f'{W}/run_info_ubipr.json', 'w') as f:
    json.dump(info, f, indent=2)
files = [f'{W}/run_info_ubipr.json', f'{W}/ubipr_download.json'] + \
        [f'{UBIPR}/{n}' for n in ('ubipr_manifest.csv', 'ubipr_skipped.csv', 'ubipr_summary.json')] + \
        sorted(glob.glob(f'{OUT}/*')) + sorted(glob.glob(f'{LOGS}/*'))
with zipfile.ZipFile(f'{W}/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in files:
        if os.path.isfile(path) and not path.endswith('.npz'):
            z.write(path, os.path.relpath(path, W))
with zipfile.ZipFile(f'{W}/results.zip') as z:
    NAMES = z.namelist()
print(f"results.zip: {len(NAMES)} files, {os.path.getsize(f'{W}/results.zip') / 2**20:.1f} MB")

In [ ]:
# 10. Final checks: the run fails here if a hard check fails
checks = [('download recorded (sha256, size)', len(DOWNLOAD['sha256']) == 64 and DOWNLOAD['size_bytes'] > 4e8,
           DOWNLOAD['sha256'][:16], True),
          ('same archive as the laptop copy of 1 Nov', DOWNLOAD['same_as_laptop_copy'], '', False),
          ('20: every image paired with an annotation', PREP['files']['images'] == PREP['files']['annotations'],
           str(PREP['files']), True),
          ('20: under 1 % of the images skipped', PREP['n_skipped'] <= 0.01 * PREP['n_images_considered'],
           f"{PREP['n_skipped']} skipped", True),
          ('20: the counts of the 1 Nov copy (10,198 crops, 10,161 usable, 254 persons)',
           (PREP['n_crops'], PREP['n_usable'], PREP['n_persons']) == (10198, 10161, 254),
           f"{PREP['n_crops']}, {PREP['n_usable']}, {PREP['n_persons']}", DOWNLOAD['same_as_laptop_copy']),
          ('20: odd classes L, even classes R', PREP['odd_classes_are_L'] and PREP['even_classes_are_R'], '', True),
          ('20: same commit', PREP['git_commit'] == COMMIT, PREP['git_commit'], True)]
probes = PREP['x1_protocol']['probes (session 2, usable)']
for label in CKPTS:
    path = f'{OUT}/verification_summary_ubipr_{label}.json'
    ok = STATUS.get(f'15_{label}') == 0 and os.path.exists(path)
    checks.append((f'15 x1 {label}: finished', ok, f'exit {STATUS.get(f"15_{label}")}', True))
    if not ok:
        continue
    s, m = load(path), pd.read_csv(f'{OUT}/verification_metrics_ubipr_{label}.csv')
    x1, row = s['per_mode']['x1'], m.iloc[0]
    checks += [(f'15 x1 {label}: every usable probe scored or counted',
                row.n_probes + x1['n_probes_without_template'] == probes,
                f"{row.n_probes} scored + {x1['n_probes_without_template']} without a template = {probes}?", True),
               (f'15 x1 {label}: one genuine comparison per probe', row.n_genuine == row.n_probes, '', True),
               (f'15 x1 {label}: subsets all, 5 distances, glasses',
                m.subset.tolist()[:1] == ['all'] and m.subset.str.startswith('cam_distance=').sum() == 5
                and m.subset.str.startswith('glasses=').sum() == 2, ', '.join(m.subset), True),
               (f'15 x1 {label}: metrics in range', m.eer.between(0, 1).all() and m.rank1.between(0, 1).all(), '', True),
               (f'15 x1 {label}: same commit', s['git_commit'] == COMMIT, s['git_commit'], True),
               (f'15 x1 {label}: in results zip', f'verification_ubipr/verification_metrics_ubipr_{label}.csv' in NAMES,
                '', True)]
checks.append(('results zip holds no images of the dataset',
               not [n for n in NAMES if n.lower().endswith(('.jpg', '.jpeg')) or '/crops/' in n
                    or 'qc_contact_sheet' in n], '', True))
for name, passed, detail, hard in checks:
    print(f"{'PASS' if passed else ('FAIL' if hard else 'WARN')}  {name}  {detail}")
failed = [c for c in checks if not c[1] and c[3]]
assert not failed, f'{len(failed)} hard check(s) failed: {[c[0] for c in failed]}'
print('all hard checks passed')

## After the run

1. Download `results.zip` from the Output panel and unzip it into a new folder under `D:\TempCLG\results`. Never commit it, and never download or share the crops (`ubipr/crops`) or the QC sheet.
2. Quote the numbers as *zero-shot cross-dataset, cross-sensor* verification (VISOB smartphone to UBIPr DSLR at 4-8 m, unseen persons), with the CIs, and cite Padole & Proença (ICB 2012) and UBIRIS.v2 (TPAMI 2010).
3. Limitations to state: UBIPr's eye label is its own (L = the eye with the nose on the crop's right) and comparisons stay within one label; 12 misfiled crops (photographs outside their person's capture run: 5 that share a photograph with another person's class, 7 from one run of an unlabelled person) are left out, and any remaining shared photograph between classes would make them one person, never impostors; the crops are re-framed from landmarks, so a different framing (`--k`, `--anchor-y` of 20) changes the numbers.